# REVE frozen heads — expanded sleep corpus (A-vig + Head C)

Private GPU kernel (prefer **T4**, not P100). Offline REVE from `muse-eeg-heads-cache` (no HF_TOKEN).

- Corpus: `vigilance_sleep_edf` N1-slice (~124 subjects, Sleep-EDF + HMC)
- Tasks: A-vig `drowsy`/`hypnagogic`; Head C `wake`/`light` (deep/REM absent on N1-slice)
- Encoder: **frozen REVE-base**; tiny linear heads only — **no fine-tune**


In [ ]:
import os, sys, json, gc, time
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Prefer T4; document accelerator
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    if 'P100' in name or 'Tesla P100' in name:
        print('WARNING: P100 detected — prefer T4 UI setting; resample uses CPU interpolate fallback')

SRC = Path('/kaggle/input/muse-eeg-heads-src')
WIN = Path('/kaggle/input/muse-eeg-heads-windows/vigilance_sleep_edf/windows')
SPLITS = Path('/kaggle/input/muse-eeg-heads-windows/vigilance_sleep_edf/splits')
CACHE = Path('/kaggle/input/muse-eeg-heads-cache')
OUT = Path('/kaggle/working/reve_sleep_heads')
OUT.mkdir(parents=True, exist_ok=True)
EMB = OUT / 'emb_cache'
EMB.mkdir(exist_ok=True)

import zipfile, shutil
src=Path('/kaggle/input/muse-eeg-heads-src')
# dir-mode zip may ship packages as *.zip
for zname in ('heads.zip',):
    zpath=src/zname
    if zpath.exists() and not (src/'heads').exists():
        print('unzip', zpath)
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(src)
print('src listing', sorted(p.name for p in src.iterdir())[:30])
# also copy flat modules into a work src if needed
work=Path('/kaggle/working/src_flat')
work.mkdir(exist_ok=True)
for p in src.glob('*.py'):
    shutil.copy2(p, work/p.name)
if (src/'heads').exists():
    dst=work/'heads'
    if dst.exists(): shutil.rmtree(dst)
    shutil.copytree(src/'heads', dst, ignore=shutil.ignore_patterns('__pycache__'))
sys.path.insert(0, str(work))
sys.path.insert(0, str(src))
print('import check…')
print('WIN exists', WIN.exists(), 'n', len(list(WIN.glob('*_windows.npz'))))
print('SPLITS', list(SPLITS.glob('*.json'))[:5])



In [ ]:
from reve_encoder import FrozenREVEEncoder
from head_c import HeadCLinear, class_weights_from_y, undersample_balanced
from heads.head_a_vig import HEAD_A_VIG_LABELS_2, HeadAVigLinear
from metrics import per_class_report, confusion_matrix

SEED=42
BATCH_ENC=64
BATCH_HEAD=256
EPOCHS=20
LR=1e-3
PATIENCE=5
MAX_TRAIN_BAL=80000
FLAT_STD=0.1
PEAK_ABS=15.0  # windows already zscored in some packs; sleep-edf is uV — use 350 if needed
# Sleep-EDF windows are roughly uV scale
PEAK_ABS=350.0
HEAD_C_LABELS=['wake','light']
TARGET_SR=256.0

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
rng=np.random.default_rng(SEED)
torch.manual_seed(SEED)

split_map={}
for sp in ('train','val','test'):
    for sid in json.loads((SPLITS/f'{sp}_subjects.json').read_text())['subjects']:
        split_map[sid]=sp
policy=json.loads((SPLITS/'split_policy.json').read_text())
print('subjects', {k:sum(1 for v in split_map.values() if v==k) for k in ('train','val','test')})


In [ ]:
def subject_of(rid, policy):
    recs=policy.get('recordings',{})
    if rid in recs:
        return recs[rid]['subject_id']
    if rid.startswith('SN'):
        return rid
    return rid[:5]

def qc_mask(X):
    std=X.std(axis=-1)
    peak=np.max(np.abs(X), axis=-1)
    return ~((std<FLAT_STD).any(axis=-1) | (peak>PEAK_ABS).any(axis=-1))

print('Loading REVE offline from cache…')
enc=FrozenREVEEncoder(source_sr=TARGET_SR, prefer_local=True, device=device,
                      cache_roots=[CACHE/'models', CACHE])
enc.to(device)
enc.eval()
print('emb_dim', enc.emb_dim, 'model_local', enc.model_from_local, 'pos_local', enc.positions_from_local)
notes=enc.adapter_notes() if hasattr(enc,'adapter_notes') else {}
print(notes)


In [ ]:
@torch.no_grad()
def encode_X(X):
    outs=[]
    for i in range(0,len(X),BATCH_ENC):
        xb=torch.from_numpy(X[i:i+BATCH_ENC]).to(device)
        outs.append(enc(xb).detach().cpu().numpy().astype(np.float32))
    return np.concatenate(outs,0) if outs else np.zeros((0,enc.emb_dim),np.float32)

rids=sorted(p.name.replace('_windows.npz','') for p in WIN.glob('*_windows.npz'))
print('recordings', len(rids))
meta_recs=[]
t0=time.time()
for i,rid in enumerate(rids):
    cache=EMB/f'{rid}_emb.npz'
    sid=subject_of(rid, policy)
    split=split_map.get(sid)
    if split is None:
        print('skip no split', rid, sid); continue
    if cache.exists():
        z=np.load(cache)
        meta_recs.append({'recording_id':rid,'subject_id':sid,'split':split,'n':int(z['emb'].shape[0])})
        continue
    data=np.load(WIN/f'{rid}_windows.npz', allow_pickle=True)
    X=data['X'].astype(np.float32)
    names=[str(x) for x in data['label_names'].tolist()]
    y_raw=data['y'].astype(np.int64)
    # map to vig labels
    name_to={n:HEAD_A_VIG_LABELS_2.index(n) for n in names if n in HEAD_A_VIG_LABELS_2}
    y_a=np.full(len(y_raw),-1,np.int64)
    for oi,n in enumerate(names):
        if n in name_to:
            y_a[y_raw==oi]=name_to[n]
    coarse=np.asarray(data['stage_coarse'], dtype=object)
    y_c=np.array([HEAD_C_LABELS.index(str(c)) if str(c) in HEAD_C_LABELS else -1 for c in coarse], dtype=np.int64)
    keep=qc_mask(X) & (y_a>=0) & (y_c>=0)
    X=X[keep]; y_a=y_a[keep]; y_c=y_c[keep]
    emb=encode_X(X)
    np.savez_compressed(cache, emb=emb, y_a=y_a, y_c=y_c, subject_id=np.asarray(sid), split=np.asarray(split), recording_id=np.asarray(rid))
    meta_recs.append({'recording_id':rid,'subject_id':sid,'split':split,'n':int(len(y_a))})
    if (i+1)%10==0 or i==0:
        rate=(i+1)/(time.time()-t0+1e-6)
        print(f'{i+1}/{len(rids)} {rid} n={len(y_a)} rate={rate:.2f} rec/s', flush=True)
    del data,X,emb; gc.collect()
print('encode done', time.time()-t0)


In [ ]:
def load_pooled(split, task):
    embs, ys=[], []
    key='y_a' if task=='a' else 'y_c'
    labels=HEAD_A_VIG_LABELS_2 if task=='a' else HEAD_C_LABELS
    detail={'counts':Counter(),'n_subjects':0}
    subs=set()
    for p in sorted(EMB.glob('*_emb.npz')):
        z=np.load(p, allow_pickle=True)
        if str(z['split'])!=split: continue
        emb=z['emb'].astype(np.float32); y=z[key].astype(np.int64)
        m=y>=0; emb,y=emb[m],y[m]
        if len(y)==0: continue
        embs.append(emb); ys.append(y)
        subs.add(str(z['subject_id']))
        detail['counts'].update({labels[j]:int((y==j).sum()) for j in range(len(labels))})
    detail['counts']=dict(detail['counts']); detail['n_subjects']=len(subs)
    return np.concatenate(embs), np.concatenate(ys), detail

def eval_head(head, emb, y, labels):
    head.eval(); preds=[]
    with torch.no_grad():
        for i in range(0,len(emb),BATCH_HEAD):
            logits=head(torch.from_numpy(emb[i:i+BATCH_HEAD]).to(device))
            preds.extend(logits.argmax(-1).cpu().numpy().tolist())
    pred=np.asarray(preds,np.int64)
    report=per_class_report(y.tolist(), pred.tolist(), list(labels))
    acc=float((pred==y).mean()) if len(y) else 0.0
    return {'n':int(len(y)),'accuracy':acc,'macro_f1':float(report['macro_f1']['f1']),
            'per_class':{k:v for k,v in report.items() if k!='macro_f1'},
            'pred_counts':{labels[i]:int((pred==i).sum()) for i in range(len(labels))},
            'true_counts':{labels[i]:int((y==i).sum()) for i in range(len(labels))}}

def train_head(emb_fit,y_fit,emb_val,y_val,labels,ctor):
    head=ctor(in_dim=emb_fit.shape[-1], n_classes=len(labels)).to(device)
    w=class_weights_from_y(y_fit, n_classes=len(labels))
    crit=nn.CrossEntropyLoss(weight=w.to(device))
    opt=torch.optim.Adam(head.parameters(), lr=LR)
    loader=DataLoader(TensorDataset(torch.from_numpy(emb_fit), torch.from_numpy(y_fit)), batch_size=BATCH_HEAD, shuffle=True)
    best=-1; best_state=None; stale=0; hist=[]
    for ep in range(EPOCHS):
        head.train(); total=n=0
        for xb,yb in loader:
            xb,yb=xb.to(device),yb.to(device)
            opt.zero_grad(); loss=crit(head(xb),yb); loss.backward(); opt.step()
            total+=float(loss.item())*len(yb); n+=len(yb)
        tr=eval_head(head,emb_fit,y_fit,labels); va=eval_head(head,emb_val,y_val,labels)
        row={'epoch':ep+1,'loss':total/max(n,1),'train_macro_f1':tr['macro_f1'],'val_macro_f1':va['macro_f1'],'val_acc':va['accuracy']}
        hist.append(row); print(row, flush=True)
        if va['macro_f1']>best+1e-4:
            best=va['macro_f1']; best_state={k:v.detach().cpu().clone() for k,v in head.state_dict().items()}; stale=0
        else:
            stale+=1
            if stale>=PATIENCE:
                print('early stop', ep+1); break
    if best_state: head.load_state_dict(best_state)
    return head, hist, best


In [ ]:
# ---- A-vig ----
emb_tr,y_tr,dtr=load_pooled('train','a')
emb_va,y_va,dva=load_pooled('val','a')
emb_te,y_te,dte=load_pooled('test','a')
print('A', dtr['counts'], dva['counts'], dte['counts'])
emb_fit,y_fit=undersample_balanced(emb_tr,y_tr,rng)
if len(y_fit)>MAX_TRAIN_BAL:
    # keep balance
    per=MAX_TRAIN_BAL//2
    picks=[]
    for c in np.unique(y_fit):
        cand=np.where(y_fit==c)[0]
        picks.append(rng.choice(cand, size=min(per,len(cand)), replace=False))
    idx=np.concatenate(picks); rng.shuffle(idx)
    emb_fit,y_fit=emb_fit[idx],y_fit[idx]
print('fit', Counter(y_fit.tolist()))
head_a,hist_a,best_a=train_head(emb_fit,y_fit,emb_va,y_va,HEAD_A_VIG_LABELS_2,
    lambda in_dim,n_classes: HeadAVigLinear(in_dim=in_dim,n_classes=n_classes))
train_a=eval_head(head_a,emb_fit,y_fit,HEAD_A_VIG_LABELS_2)
val_a=eval_head(head_a,emb_va,y_va,HEAD_A_VIG_LABELS_2)
test_a=eval_head(head_a,emb_te,y_te,HEAD_A_VIG_LABELS_2)
torch.save({'state_dict':head_a.state_dict(),'label_list':HEAD_A_VIG_LABELS_2,'encoder':'REVE_frozen'}, OUT/'head_a_vig_reve_linear.pt')
print('A-vig val', val_a['macro_f1'], 'test', test_a['macro_f1'])


In [ ]:
# ---- Head C ----
emb_tr,y_tr,dtr=load_pooled('train','c')
emb_va,y_va,dva=load_pooled('val','c')
emb_te,y_te,dte=load_pooled('test','c')
emb_fit,y_fit=undersample_balanced(emb_tr,y_tr,rng)
if len(y_fit)>MAX_TRAIN_BAL:
    per=MAX_TRAIN_BAL//2
    picks=[]
    for c in np.unique(y_fit):
        cand=np.where(y_fit==c)[0]
        picks.append(rng.choice(cand, size=min(per,len(cand)), replace=False))
    idx=np.concatenate(picks); rng.shuffle(idx)
    emb_fit,y_fit=emb_fit[idx],y_fit[idx]
head_c,hist_c,best_c=train_head(emb_fit,y_fit,emb_va,y_va,HEAD_C_LABELS,
    lambda in_dim,n_classes: HeadCLinear(in_dim=in_dim,n_classes=n_classes))
train_c=eval_head(head_c,emb_fit,y_fit,HEAD_C_LABELS)
val_c=eval_head(head_c,emb_va,y_va,HEAD_C_LABELS)
test_c=eval_head(head_c,emb_te,y_te,HEAD_C_LABELS)
torch.save({'state_dict':head_c.state_dict(),'label_list':HEAD_C_LABELS,'encoder':'REVE_frozen','note':'wake_light_2way_n1slice'}, OUT/'head_c_wake_light_reve_linear.pt')
print('Head C val', val_c['macro_f1'], 'test', test_c['macro_f1'])


In [ ]:
from datetime import datetime, timezone
created=datetime.now(timezone.utc).isoformat()
# ship bars (honest)
a_ship = (test_a['macro_f1']>=0.65 and val_a['macro_f1']>=0.60
          and all(test_a['per_class'][n]['f1']>=0.50 for n in HEAD_A_VIG_LABELS_2)
          and all(test_a['pred_counts'].get(n,0)>0 for n in HEAD_A_VIG_LABELS_2))
c_ship=False  # N1-slice 2-way only
summary={
  'step':'reve_sleep_heads_kaggle',
  'completed_utc':created,
  'device':str(device),
  'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
  'encoder':'REVE_frozen',
  'backbone_finetuned':False,
  'head_a_vig':{'ship_candidate':bool(a_ship),'val_macro_f1':val_a['macro_f1'],'test_macro_f1':test_a['macro_f1'],
                'val_acc':val_a['accuracy'],'test_acc':test_a['accuracy'],
                'test_per_class_f1':{k:test_a['per_class'][k]['f1'] for k in HEAD_A_VIG_LABELS_2},
                'val':val_a,'test':test_a},
  'head_c':{'ship_candidate':False,'task':'wake_light_2way_n1slice',
            'val_macro_f1':val_c['macro_f1'],'test_macro_f1':test_c['macro_f1'],
            'test_per_class_f1':{k:test_c['per_class'][k]['f1'] for k in HEAD_C_LABELS},
            'val':val_c,'test':test_c},
  'compare_note':'Pull alongside exports/train_heads_expanded_sleep_summary.json (CBraMod).',
}
(OUT/'metrics_summary.json').write_text(json.dumps(summary, indent=2)+'
')
(OUT/'step_summary.json').write_text(json.dumps({k:summary[k] for k in ('step','completed_utc','head_a_vig','head_c','encoder','backbone_finetuned')}, indent=2)+'
')
print(json.dumps(summary, indent=2))
